# DIAGNOSTIC: Baseline Model Check
Test if positive R² is theoretically achievable with this data

**Approach:**
1. Ridge Regression (simplest possible linear model)
2. If Ridge gets R² > -0.005 → Signal exists, tree models should work better
3. If Ridge gets R² ≈ -0.01 or worse → Problem has ceiling, not a model issue

**Why Ridge?** No overfitting risk, pure generalization test

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("BASELINE DIAGNOSTIC: Ridge Regression Test")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/3] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== TEST RIDGE WITH DIFFERENT ALPHAS ==============
print("\n[2/3] Testing Ridge Regression with different regularization strengths...\n")

alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
ridge_results = []

for alpha in alphas:
    print(f"Testing Ridge with alpha={alpha}...")
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    test_preds_ridge = np.zeros(len(X_test))
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # Standardize features (important for Ridge)
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        model = Ridge(alpha=alpha)
        model.fit(X_tr_scaled, y_tr)
        
        test_preds_ridge += model.predict(X_test_scaled) / 5
        r2 = r2_score(y_val, model.predict(X_val_scaled))
        fold_scores.append(r2)
        
        del model, scaler, X_tr_scaled, X_val_scaled, X_test_scaled, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    ridge_results.append({
        'alpha': alpha,
        'cv_r2': avg_r2,
        'best_fold': max(fold_scores),
        'worst_fold': min(fold_scores)
    })
    print(f"  → CV R² (5-fold): {avg_r2:.6f} (range: {min(fold_scores):.6f} to {max(fold_scores):.6f})")

results_df = pd.DataFrame(ridge_results)
best_alpha_idx = results_df['cv_r2'].idxmax()
best_alpha = results_df.loc[best_alpha_idx, 'alpha']
best_ridge_cv = results_df.loc[best_alpha_idx, 'cv_r2']

print(f"\n" + "="*70)
print(f"RIDGE REGRESSION RESULTS:")
print(f"="*70)
print(results_df.to_string(index=False))
print(f"\n✓ BEST Ridge: alpha={best_alpha} → CV R² = {best_ridge_cv:.6f}")

# ============== TRAIN FINAL RIDGE MODEL ==============
print(f"\n[3/3] Training final Ridge model (full data)...")

scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled = scaler_final.transform(X_test)

ridge_final = Ridge(alpha=best_alpha)
ridge_final.fit(X_train_scaled, y_train)

ensemble_preds = ridge_final.predict(X_test_scaled)

# ============== CREATE SUBMISSION ==============
print(f"\nCreating submission...")
submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

submission.to_csv('submission_ridge_baseline.csv', index=False)
print(f"✓ Saved: submission_ridge_baseline.csv")

print("\n" + "="*70)
print("RIDGE BASELINE - DIAGNOSTIC COMPLETE")
print("="*70)
print(f"\n📊 INTERPRETATION GUIDE:")
print(f"\nIf Ridge CV R² > -0.005:")
print(f"  → Signal exists! Tree models (XGBoost) should beat linear")
print(f"  → Keep working with 150-feature XGBoost approach")
print(f"\nIf Ridge CV R² ≤ -0.01:")
print(f"  → Problem has hard ceiling, not a model issue")
print(f"  → Consider: different features, target engineering, or accept -0.01156")
print(f"\n" + "="*70)